# 🤖 03 — Modelado y Evaluación
**Proyecto:** Detección de Fraude Financiero con ML + LLM Explicativo  
**Responsable:** Nombre Completo 2 · correo2@eafit.edu.co

---
**Objetivo:** Entrenar y comparar Baseline (LR) → Experimento 1 (Random Forest) → Experimento 2 (XGBoost). Evaluar en test set, generar curvas ROC y análisis SHAP.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

plt.rcParams['figure.dpi'] = 130
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
os.makedirs('../figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('✅ Imports OK')

## 1. Cargar datos procesados

In [ ]:
PROC = '../data/processed/'

if os.path.exists(PROC + 'X_train.npy'):
    X_train = np.load(PROC + 'X_train.npy')
    X_val   = np.load(PROC + 'X_val.npy')
    X_test  = np.load(PROC + 'X_test.npy')
    y_train = np.load(PROC + 'y_train.npy')
    y_val   = np.load(PROC + 'y_val.npy')
    y_test  = np.load(PROC + 'y_test.npy')
    print('✅ Datos cargados desde data/processed/')
else:
    print('⚠️  Archivos procesados no encontrados. Ejecuta primero 02_preprocessing.ipynb')
    print('   Generando datos sintéticos de emergencia...')
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    X_all, y_all = make_classification(
        n_samples=10_000, n_features=30, n_informative=15,
        weights=[0.9965, 0.0035], random_state=RANDOM_STATE
    )
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_all, y_all, test_size=0.3, stratify=y_all, random_state=RANDOM_STATE)
    X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_tr)
    X_val   = sc.transform(X_val)
    X_test  = sc.transform(X_test)
    y_train = y_tr

spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')
print(f'Fraude en train: {y_train.mean()*100:.2f}%  |  scale_pos_weight para XGB: {spw:.1f}')

## 2. Función de evaluación unificada

In [ ]:
resultados_val  = []
resultados_test = []

def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_ev, y_ev, conjunto='val'):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_ev)
    y_prob = modelo.predict_proba(X_ev)[:, 1] if hasattr(modelo, 'predict_proba') else None
    metricas = {
        'Modelo':     nombre,
        'Accuracy':   round(accuracy_score(y_ev, y_pred), 4),
        'Precision':  round(precision_score(y_ev, y_pred, zero_division=0), 4),
        'Recall':     round(recall_score(y_ev, y_pred, zero_division=0), 4),
        'F1':         round(f1_score(y_ev, y_pred, zero_division=0), 4),
        'AUC-ROC':    round(roc_auc_score(y_ev, y_prob), 4) if y_prob is not None else None,
        '_modelo':    modelo,
        '_y_prob':    y_prob,
        '_y_pred':    y_pred,
    }
    emoji = '🏆' if metricas['AUC-ROC'] and metricas['AUC-ROC'] > 0.95 else '  '
    print(f"{emoji} {nombre:<35s}  F1={metricas['F1']:.4f}  AUC={metricas['AUC-ROC']:.4f}")
    return metricas

print('── Entrenando modelos en validación ────────────────────')

## 3. Baseline — Regresión Logística

In [ ]:
lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=0.01,
    solver='lbfgs',
    random_state=RANDOM_STATE
)
resultados_val.append(evaluar_modelo('Baseline — Log. Regression', lr, X_train, y_train, X_val, y_val))

## 4. Experimento 1 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
resultados_val.append(evaluar_modelo('Exp. 1 — Random Forest', rf, X_train, y_train, X_val, y_val))

## 5. Experimento 2 — XGBoost (modelo principal)

In [ ]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=spw,
    reg_alpha=0.1,
    reg_lambda=1.5,
    eval_metric='aucpr',
    early_stopping_rounds=40,
    random_state=RANDOM_STATE,
    verbosity=0
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
resultados_val.append(evaluar_modelo('Exp. 2 — XGBoost ✓', xgb, X_train, y_train, X_val, y_val))

## 6. Tabla comparativa en validación

In [ ]:
cols_display = ['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
tabla_val = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in resultados_val])
display(tabla_val.style
    .highlight_max(subset=['Accuracy','Precision','Recall','F1','AUC-ROC'], color='#d4edda')
    .format({c: '{:.4f}' for c in ['Accuracy','Precision','Recall','F1','AUC-ROC']})
    .set_caption('Comparación de modelos — Conjunto de Validación')
)

## 7. Validación cruzada del mejor modelo

In [ ]:
print('── Cross-validation 5-fold estratificado (XGBoost) ────')
X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
xgb_cv = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.03,
    subsample=0.85, colsample_bytree=0.85,
    scale_pos_weight=spw, random_state=RANDOM_STATE, verbosity=0
)
scores_auc = cross_val_score(xgb_cv, X_full, y_full, cv=skf, scoring='roc_auc', n_jobs=-1)
scores_f1  = cross_val_score(xgb_cv, X_full, y_full, cv=skf, scoring='f1', n_jobs=-1)

print(f'  AUC-ROC: {scores_auc.mean():.4f} ± {scores_auc.std():.4f}  |  folds: {np.round(scores_auc,4)}')
print(f'  F1:      {scores_f1.mean():.4f} ± {scores_f1.std():.4f}  |  folds: {np.round(scores_f1,4)}')

## 8. Evaluación final en TEST SET

In [ ]:
# Elegir el mejor modelo de validación
mejor = max(resultados_val, key=lambda r: r['AUC-ROC'])
modelo_final = mejor['_modelo']

y_pred_test = modelo_final.predict(X_test)
y_prob_test = modelo_final.predict_proba(X_test)[:, 1]

acc_test = accuracy_score(y_test, y_pred_test)
f1_test  = f1_score(y_test, y_pred_test)
auc_test = roc_auc_score(y_test, y_prob_test)
prec_test = precision_score(y_test, y_pred_test)
rec_test  = recall_score(y_test, y_pred_test)

print(f'══ RESULTADOS FINALES EN TEST SET — {mejor["Modelo"]} ══')
print(f'  Accuracy  = {acc_test:.4f}')
print(f'  Precision = {prec_test:.4f}')
print(f'  Recall    = {rec_test:.4f}')
print(f'  F1-Score  = {f1_test:.4f}')
print(f'  AUC-ROC   = {auc_test:.4f}')
print()
print(classification_report(y_test, y_pred_test, target_names=['Legítima','Fraude'], digits=4))

## 9. Visualizaciones de evaluación

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Matriz de confusión ---
cm = confusion_matrix(y_test, y_pred_test)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels = np.array([[f'{v}\n({p:.1%})' for v, p in zip(row_v, row_p)]
                   for row_v, row_p in zip(cm, cm_norm)])
sns.heatmap(cm_norm, annot=labels, fmt='', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Legítima', 'Pred: Fraude'],
            yticklabels=['Real: Legítima', 'Real: Fraude'],
            linewidths=1.5, linecolor='white', vmin=0, vmax=1)
axes[0].set_title(f'Matriz de Confusión\n{mejor["Modelo"]}', fontweight='bold')

# --- Curva ROC ---
RocCurveDisplay.from_predictions(
    y_test, y_prob_test, ax=axes[1], color='#003d79',
    name=f'{mejor["Modelo"]} (AUC={auc_test:.3f})'
)
# Agregar baseline y otros modelos
for r in resultados_val:
    if r['Modelo'] != mejor['Modelo'] and r['_y_prob'] is not None:
        y_prob_r = r['_modelo'].predict_proba(X_test)[:, 1]
        auc_r = roc_auc_score(y_test, y_prob_r)
        RocCurveDisplay.from_predictions(
            y_test, y_prob_r, ax=axes[1],
            name=f'{r["Modelo"][:20]} ({auc_r:.3f})', alpha=0.6
        )
axes[1].plot([0,1],[0,1],'--', color='gray', alpha=0.5, label='Random')
axes[1].set_title('Curvas ROC — Test Set', fontweight='bold')
axes[1].legend(fontsize=7.5)

# --- Precision-Recall ---
PrecisionRecallDisplay.from_predictions(
    y_test, y_prob_test, ax=axes[2], color='#9b1b1b',
    name=mejor['Modelo']
)
axes[2].set_title('Curva Precision-Recall', fontweight='bold')

plt.suptitle('Evaluación final — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/curvas_evaluacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: figures/curvas_evaluacion.png')

## 10. Importancia de features — SHAP

In [ ]:
try:
    import shap
    feat_names_path = '../data/processed/feature_names.csv'
    if os.path.exists(feat_names_path):
        feat_names = pd.read_csv(feat_names_path).iloc[:, 0].tolist()
    else:
        feat_names = [f'feature_{i}' for i in range(X_test.shape[1])]

    explainer = shap.TreeExplainer(modelo_final)
    n_sample  = min(500, X_test.shape[0])
    shap_vals = explainer.shap_values(X_test[:n_sample])

    plt.figure(figsize=(10, 7))
    shap.summary_plot(
        shap_vals, X_test[:n_sample],
        feature_names=feat_names[:X_test.shape[1]],
        show=False, plot_size=None
    )
    plt.title('Importancia de features — SHAP values (XGBoost)', fontweight='bold', pad=14)
    plt.tight_layout()
    plt.savefig('../figures/shap_importancia.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Guardado: figures/shap_importancia.png')
except ImportError:
    print('⚠️  SHAP no instalado. Instalar con: pip install shap')
except Exception as e:
    print(f'⚠️  SHAP no disponible: {e}')

## 11. Guardar modelo final

In [ ]:
joblib.dump(modelo_final, '../models/modelo_final.joblib')

# Guardar métricas finales
metricas_finales = {
    'modelo': mejor['Modelo'],
    'accuracy': acc_test, 'precision': prec_test,
    'recall': rec_test, 'f1': f1_test, 'auc_roc': auc_test
}
pd.Series(metricas_finales).to_json('../models/metricas_finales.json')

print('✅ Modelo guardado: models/modelo_final.joblib')
print('✅ Métricas guardadas: models/metricas_finales.json')
print('\n→ Continuar con: 04_llm_explicaciones.ipynb')